# HC-3 Download And Phase Target Prototype

This notebook starts the GRTPL workflow:

1. Download public CRCNS HC-3 documentation and metadata.
2. Inspect channel-order metadata before choosing adjacent channel pairs.
3. Download a small real HC-3 LFP session from the Buzsaki Lab mirror.
4. Build a two-channel differential LFP trace and phase targets from real data.

Full HC-3 LFP data is large. Keep raw data under `data/raw/`, which is ignored by git.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

project_root

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from zipfile import ZipFile

from grtpl.hc3 import download_public_hc3_docs, download_buzsaki_hc3_session_files, load_eeg_channels, read_hc3_xml_metadata
from grtpl.signal import causal_bandpass, acausal_bandpass, hilbert_phase, decimate_by_timestamp
from grtpl.targets import find_next_phase_crossings, next_crossing_for_samples, nominal_phase_targets

pio.templates.default = "plotly_dark"

## Download Public HC-3 Docs

These files are small enough to fetch immediately and include the data description, metadata tables, channel order, and probe geometry.

In [ ]:
raw_dir = project_root / "data" / "raw"
downloads = download_public_hc3_docs(raw_dir)
pd.DataFrame([d.__dict__ for d in downloads])

## Inspect Channel Metadata

The next practical decision is selecting adjacent hippocampal channels. Start by listing the channel-order files.

In [ ]:
channel_zip = raw_dir / "hc3" / "docs" / "crcns-hc3-channelorder.zip"
with ZipFile(channel_zip) as zf:
    channel_files = zf.namelist()

channel_files[:20], len(channel_files)

## Inspect Session Metadata

The HC-3 metadata tables include a file/session table. The CSV files do not all include headers, so this first pass keeps them raw until we map columns from the data description.

In [ ]:
metadata_zip = raw_dir / "hc3" / "docs" / "crcns-hc3-metadata-tables.zip"
with ZipFile(metadata_zip) as zf:
    metadata_files = [name for name in zf.namelist() if name.endswith((".csv", ".txt"))]

metadata_files

In [ ]:
with ZipFile(metadata_zip) as zf:
    with zf.open("crcns-hc3-metadata-tables/hc3-file.csv") as handle:
        hc3_file_table = pd.read_csv(handle, header=None)

hc3_file_table.head()

## Download A Small Real HC-3 Session

`ec013.544` is a small Mwheel session from top directory `ec013.33`. The metadata reports a 29.1 second duration, and the Buzsaki Lab mirror exposes a 4.7 MB `.eeg` LFP file plus its `.xml` metadata.

In [ ]:
topdir = "ec013.33"
session = "ec013.544"

session_downloads = download_buzsaki_hc3_session_files(topdir, session, raw_dir, extensions=("xml", "eeg"))
pd.DataFrame([d.__dict__ for d in session_downloads])

## Load Two Related Channels

The XML reports 65 interleaved int16 channels sampled at 1250 Hz for LFP. For this first pass, use adjacent channels 38 and 39 from anatomical group/shank 5. The channel-order metadata for the `ec013.540_561` range marks channel 39 as the max-ripple site for shank 5, so channels 38 and 39 are a reasonable related local pair.

In [ ]:
session_dir = raw_dir / "hc3" / topdir / session
xml_path = session_dir / f"{session}.xml"
eeg_path = session_dir / f"{session}.eeg"
metadata = read_hc3_xml_metadata(xml_path)

sample_rate_hz = metadata["lfp_sampling_rate_hz"]
n_channels = metadata["n_channels"]
selected_channels = [38, 39]
channel_lfp = load_eeg_channels(eeg_path, n_channels=n_channels, channels=selected_channels)
lfp = channel_lfp[:, 1] - channel_lfp[:, 0]
timestamps = np.arange(len(lfp)) / sample_rate_hz

pd.DataFrame({
    "selected_channel": selected_channels,
    "group_index": [4, 4],
    "note": ["adjacent to max-ripple channel", "max-ripple channel in ec013.540_561 metadata"],
})

## Real-Data Phase Target Build

Use the differential LFP trace, causal 6-10 Hz filtering for the online phase estimate, and acausal 6-10 Hz filtering for offline target extraction.

In [ ]:
theta_hz = 8.0

causal_theta, sos, zf = causal_bandpass(lfp, sample_rate_hz, band_hz=(6.0, 10.0), order=4)
reference_theta = acausal_bandpass(lfp, sample_rate_hz, band_hz=(6.0, 10.0), order=4)

causal_phase = hilbert_phase(causal_theta)
reference_phase = hilbert_phase(reference_theta)

input_rate_hz = 25.0
input_timestamps, input_phase = decimate_by_timestamp(timestamps, causal_phase, input_rate_hz)

crossing_timestamps, crossing_indices = find_next_phase_crossings(timestamps, reference_phase, target_phase_rad=np.pi)
target_timestamps = next_crossing_for_samples(input_timestamps, crossing_timestamps, max_horizon_s=0.5)
target_phase = nominal_phase_targets(input_phase, input_timestamps, target_timestamps, nominal_frequency_hz=theta_hz)
target_table = pd.DataFrame({
    "input_time_s": input_timestamps,
    "causal_phase_deg": np.rad2deg(input_phase) % 360,
    "target_time_s": target_timestamps,
    "target_nominal_causal_phase_deg": np.rad2deg(target_phase) % 360,
    "lead_time_ms": (target_timestamps - input_timestamps) * 1000,
})
target_table = target_table.dropna().reset_index(drop=True)

target_table.head(12)

In [ ]:
plot_start_s = 0.0
plot_end_s = min(4.0, timestamps[-1])
window = (timestamps >= plot_start_s) & (timestamps <= plot_end_s)
phase_window = (target_table["input_time_s"] >= plot_start_s) & (target_table["input_time_s"] <= plot_end_s)
crossing_window = (crossing_timestamps >= plot_start_s) & (crossing_timestamps <= plot_end_s)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Differential LFP and theta filters", "Causal phase tokens and target nominal causal phase"),
)

fig.add_trace(go.Scatter(x=timestamps[window], y=lfp[window], mode="lines", name="differential LFP ch39 - ch38", line=dict(color="lightgray")), row=1, col=1)
fig.add_trace(go.Scatter(x=timestamps[window], y=causal_theta[window], mode="lines", name="causal 6-10 Hz"), row=1, col=1)
fig.add_trace(go.Scatter(x=timestamps[window], y=reference_theta[window], mode="lines", name="acausal reference", line=dict(dash="dot")), row=1, col=1)

for target_time in crossing_timestamps[crossing_window]:
    fig.add_vline(x=float(target_time), line_color="rgba(220, 20, 60, 0.30)", line_width=1, row=1, col=1)
    fig.add_vline(x=float(target_time), line_color="rgba(220, 20, 60, 0.20)", line_width=1, row=2, col=1)

visible_targets = target_table.loc[phase_window].copy()
fig.add_trace(
    go.Scatter(
        x=visible_targets["input_time_s"],
        y=visible_targets["causal_phase_deg"],
        mode="lines+markers",
        name="decimated causal phase at input time",
        hovertemplate="input=%{x:.4f}s<br>causal phase=%{y:.1f} deg<extra></extra>",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=visible_targets["input_time_s"],
        y=visible_targets["target_nominal_causal_phase_deg"],
        mode="markers",
        name="target nominal causal phase at input time",
        marker=dict(size=8, symbol="diamond", color="crimson"),
        customdata=np.stack([visible_targets["target_time_s"], visible_targets["lead_time_ms"]], axis=1),
        hovertemplate="input=%{x:.4f}s<br>target nominal phase=%{y:.1f} deg<br>acausal 180 target=%{customdata[0]:.4f}s<br>lead=%{customdata[1]:.1f} ms<extra></extra>",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=crossing_timestamps[crossing_window],
        y=np.full(np.count_nonzero(crossing_window), 180.0),
        mode="markers",
        name="acausal 180 deg target time",
        marker=dict(size=9, symbol="x", color="white"),
        hovertemplate="acausal 180 target=%{x:.4f}s<br>reference phase=180 deg<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.update_yaxes(title_text="signal", row=1, col=1)
fig.update_yaxes(title_text="phase deg", range=[-10, 370], row=2, col=1)
fig.update_xaxes(title_text="time s", row=2, col=1)
fig.update_layout(height=760, hovermode="x unified", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0))
fig.show()